In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
%matplotlib inline

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (8, 4.5)

df = pd.read_csv("popular_anime.csv")
df["year"] = pd.to_datetime(df["aired_from"], errors="coerce", utc=True).dt.year
df = df.dropna(subset=["year", "type"])
df["year"] = df["year"].astype(int)
df["decade"] = df["year"] // 10 * 10
df = df[df["decade"].between(1970, 2020)]

print("Shape :", df.shape)
print("Decades :", sorted(df["decade"].unique()))

In [ ]:
# ---------- Stacked bar chart : composition of types by decade ----------
types = ["TV", "Movie", "OVA", "ONA", "Special"]
comp = df[df["type"].isin(types)].pivot_table(
    index="decade", columns="type", values="id", aggfunc="count").fillna(0)
comp = comp[types]

fig, ax = plt.subplots()
bottom = np.zeros(len(comp))
colors = ["#2a6fdb", "#4c9f70", "#e07a5f", "#f2cc8f", "#9c89b8"]

for t, col in zip(types, colors):
    ax.bar(comp.index.astype(str), comp[t], bottom=bottom, label=t, color=col,
           edgecolor="white", width=0.7)
    bottom += comp[t].values

ax.set_title("Composition of Content Types Across Decades")
ax.set_xlabel("Decade")
ax.set_ylabel("Number of titles")
ax.legend(title="Type")
plt.tight_layout()
plt.show()

In [ ]:
# ---------- Grouped bar chart : average score by type across decades ----------
avg = df[df["type"].isin(["TV", "Movie", "OVA"])].pivot_table(
    index="decade", columns="type", values="score", aggfunc="mean")

x = np.arange(len(avg))
w = 0.26

fig, ax = plt.subplots()
for i, (t, col) in enumerate(zip(["TV", "Movie", "OVA"], colors)):
    ax.bar(x + (i - 1) * w, avg[t], width=w, label=t, color=col, edgecolor="black")

ax.set_xticks(x, avg.index.astype(str))
ax.set_ylim(5.5, 7.5)
ax.set_title("Average Score by Type Across Decades")
ax.set_xlabel("Decade")
ax.set_ylabel("Average score")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ---------- Stacked area chart : growth of each type over the years ----------
area = (df[(df["type"].isin(types)) & (df["year"].between(1990, 2024))]
        .pivot_table(index="year", columns="type", values="id", aggfunc="count")
        .fillna(0)[types])

fig, ax = plt.subplots()
ax.stackplot(area.index, [area[t] for t in types], labels=types,
             colors=colors, alpha=0.85)

ax.set_title("Growth of Each Content Type Over the Years")
ax.set_xlabel("Year")
ax.set_ylabel("Number of titles")
ax.legend(loc="upper left", title="Type")
ax.set_xlim(1990, 2024)
plt.tight_layout()
plt.show()

In [ ]:
# ---------- Heat map : average score by decade and type ----------
heat = df[df["type"].isin(types)].pivot_table(
    index="type", columns="decade", values="score", aggfunc="mean").loc[types]

fig, ax = plt.subplots(figsize=(8, 3.8))
im = ax.imshow(heat, cmap="YlGnBu", aspect="auto")

ax.set_xticks(range(len(heat.columns)), heat.columns)
ax.set_yticks(range(len(heat.index)), heat.index)

for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        v = heat.iloc[i, j]
        if not np.isnan(v):
            ax.text(j, i, round(v, 1), ha="center", va="center", fontsize=9)

ax.set_title("Average Score by Type and Decade")
fig.colorbar(im, label="Average score")
ax.grid(False)
plt.tight_layout()
plt.show()

In [ ]:
# ---------- Twin axis plot : count and average score together ----------
yearly = df[df["year"].between(1995, 2024)].groupby("year").agg(
    titles=("id", "count"), avg_score=("score", "mean"))

fig, ax1 = plt.subplots()
ax1.bar(yearly.index, yearly["titles"], color="#a8dadc", edgecolor="#457b9d",
        label="Titles released")
ax1.set_xlabel("Year")
ax1.set_ylabel("Number of titles", color="#457b9d")
ax1.tick_params(axis="y", labelcolor="#457b9d")

ax2 = ax1.twinx()
ax2.plot(yearly.index, yearly["avg_score"], color="#d1495b", linewidth=2.5,
         marker="o", markersize=4, label="Average score")
ax2.set_ylabel("Average score", color="#d1495b")
ax2.tick_params(axis="y", labelcolor="#d1495b")
ax2.grid(False)

fig.suptitle("Titles Released and Average Score by Year")
plt.tight_layout()
plt.show()

In [ ]:
# ---------- Horizontal bar chart with a colour map : score by genre ----------
g = df.dropna(subset=["genres", "score"]).copy()
g["genre"] = g["genres"].str.split(", ")
g = g.explode("genre")

genre_score = (g.groupby("genre")["score"].agg(["count", "mean"])
                 .query("count >= 300").sort_values("mean").tail(12))

norm = (genre_score["mean"] - genre_score["mean"].min()) / \
       (genre_score["mean"].max() - genre_score["mean"].min())

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(genre_score.index, genre_score["mean"],
        color=cm.viridis(norm), edgecolor="black")

for i, (v, n) in enumerate(zip(genre_score["mean"], genre_score["count"])):
    ax.text(v + 0.02, i, "%.2f  (n=%d)" % (v, n), va="center", fontsize=8)

ax.set_xlim(5.5, 8.0)
ax.set_title("Average Score by Genre  (genres with at least 300 titles)")
ax.set_xlabel("Average score")
plt.tight_layout()
plt.show()

In [ ]:
# ---------- Annotated plot ----------
peak_year = int(yearly["titles"].idxmax())
peak_val  = int(yearly["titles"].max())

fig, ax = plt.subplots()
ax.plot(yearly.index, yearly["titles"], color="#2a6fdb", linewidth=2.5)
ax.fill_between(yearly.index, yearly["titles"], alpha=0.2, color="#2a6fdb")

ax.annotate("Peak : %d titles in %d" % (peak_val, peak_year),
            xy=(peak_year, peak_val),
            xytext=(peak_year - 12, peak_val - 100),
            arrowprops={"arrowstyle": "->", "color": "black", "linewidth": 1.4},
            fontsize=10,
            bbox={"boxstyle": "round,pad=0.4", "facecolor": "#f2cc8f",
                  "edgecolor": "black"})

ax.set_title("Annual Releases with the Peak Year Highlighted")
ax.set_xlabel("Year")
ax.set_ylabel("Number of titles")
plt.tight_layout()
plt.show()

In [ ]:
# ---------- Full customisation and saving the figure ----------
fig, ax = plt.subplots(figsize=(9, 5))

for t, col in zip(["TV", "Movie", "OVA"], ["#2a6fdb", "#4c9f70", "#e07a5f"]):
    sub = df[(df["type"] == t) & (df["year"].between(1995, 2024))]
    line = sub.groupby("year")["score"].mean()
    ax.plot(line.index, line.values, label=t, color=col, linewidth=2.2, marker="o",
            markersize=3.5)

ax.set_title("Average Score Trend by Content Type", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Year", fontsize=11)
ax.set_ylabel("Average score", fontsize=11)
ax.set_xlim(1995, 2024)
ax.set_ylim(5.0, 8.0)
ax.set_xticks(range(1995, 2025, 5))
ax.legend(title="Content type", frameon=True, loc="lower right")
ax.grid(alpha=0.35, linestyle="--")

plt.tight_layout()
plt.savefig("anime_score_trend.png", dpi=200, bbox_inches="tight")
plt.show()

print("High resolution figure saved as anime_score_trend.png")